# Schlüsselaustausch Simulation

##### Klassen importieren

In [1]:
%classpath bin
%classpath ../bin
import dh.*;
import dh.KeyExchange.*;
import dh.Parameters;

## Schlüsselaustausch ohne Angriff

In [2]:
new KeyExchange()
            .setAttackType(AttackType.NONE)
            .setValidationMode(ValidationMode.NONE)
            .run();

Starte Diffie-Hellman-Schlüsselaustausch...
Angriffstyp: NONE
Validierungsmodus: NONE
Verwendete Parameter: g = 2, p = 11338529121281401305247593927686837472167231814336176336356446222468997273779837312778314348881736768689915676834236252590945754710319437953487161311319495

-> Keine Validierung der Gruppenparameter durchgeführt.

-> Keine Validierung der privaten Schlüssel durchgeführt.

[Schlüsselerzeugung]
  • Alices öffentlicher Wert A: 4ccab2fbee51b1254589de44...
  • Bobs öffentlicher Wert B:   659623a796ba0ec6490fd04f...

-> Keine Validierung der öffentlichen Werte durchgeführt.

[Ergebnisse]
  • Alices rohes K: cf4682db8b622b115bcaa0bd...
  • Bobs rohes K:   cf4682db8b622b115bcaa0bd...
  • Alices Sitzungsschlüssel: 685b4032674d78ec357b762746ac6a188d72700a696697d7467a5a2f4e4b83db
  • Bobs Sitzungsschlüssel:   685b4032674d78ec357b762746ac6a188d72700a696697d7467a5a2f4e4b83db

==> Ergebnis: Sitzungsschlüssel stimmen überein? true



## Schlüsselaustausch mit Angriff

Angriff 1 - Geändertes g, um eine kleine Untergruppe zu erzwingen

@krushalipatel2604 angriff erklären 

Fall 1: Ungeeignete bzw. manipulierte Gruppenparameter <br>
Angriff 1: Small Subgroup Attack (Manipulation öffentlicher Werte)

Der Angreifer fängt den echten öffentlichen Schlüssel auf dem Übertragungsweg ab und ersetzt ihn durch eine böswillige Zahl (z. B. $p-1$). Dadurch wird das berechnete gemeinsame Geheimnis in eine winzige Untergruppe gezwungen – es hat danach oft nur noch 2 mögliche Werte, die der Angreifer mühelos durchprobieren kann.


##### Keine Validierung

In [ ]:
new KeyExchange()
            .setAttackType(AttackType.ATTACK_1)
            .setValidationMode(ValidationMode.NONE)
            .run();

##### Validierung: Parameterprüfung

Die Parameterprüfung überprüft den verwendeten Modulus (p) und Generator (g) und bricht ab, wenn ungeeignete Werte gewählt wurden.

* **Modulus p**: 
  * *Prüfung:* Primzahlcheck (isProbablePrime) & Bitlängenprüfung (>= 512 Bit).
  * *Grund:* Zusammengesetzte Zahlen zerstören die Gruppenstruktur und vereinfachen die Schlüsselerrechnung (Angriff 1).
* **Generator g**:
  * *Prüfung:* 1 < g < p - 1.
  * *Grund:* Schließt triviale Werte (0, 1, p-1) aus.

In [ ]:
new KeyExchange()
            .setAttackType(AttackType.ATTACK_1)
            .setValidationMode(ValidationMode.VALIDATE_PARAMETERS)
            .run();

### Angriff 2 - Ungeeignetes p gewählt
@krushalipatel2604 angriff erklären
Der Angreifer sorgt dafür, dass der Modulus $p$ keine sichere Primzahl ist, sondern $p-1$ in viele kleine Primfaktoren zerfällt (eine „glatte“ Zahl). Mit dem Pohlig-Hellman-Algorithmus kann er das riesige Logarithmus-Problem in viele kleine Teilprobleme zerlegen und den kompletten privaten Schlüssel in Sekundenbruchteilen berechnen.


##### Keine Validierung

In [ ]:
new KeyExchange()
            .setAttackType(AttackType.ATTACK_2)
            .setValidationMode(ValidationMode.NONE)
            .run();

##### Validierung: Öffentliche Werte prüfen
Die öffentlichen Werte, die Bob und Alice errechnen und der jeweiligen anderen Person über den unsicheren Kanal schicken, werden überprüft, bevor damit weitergerechnet wird. Der Austausch bricht auch hier bei unsicheren Werten ab.

* **Reichweiten- & Untergruppenschutz**: 1 < Y < p - 1.
  * *Grund:* 
    * Y <= 1: Erzwingt das triviale Geheimnis K = 1.
    * Y >= p - 1: Kompromittiert das Geheimnis auf K in {1, p-1} (Angriff 2).
    * Y >= p: Liegt außerhalb des Restklassenrings.

In [ ]:
new KeyExchange()
            .setAttackType(AttackType.ATTACK_2)
            .setValidationMode(ValidationMode.VALIDATE_PUBLIC_VALUES)
            .run();

### Angriff 3 - Statischer Schlüssel + CRT
@krushalipatel2604 angriff erklären
Verwendet ein Teilnehmer denselben privaten Schlüssel für viele Verbindungen, baut der Angreifer nacheinander mehrere Sitzungen auf und schickt jedes Mal einen anderen manipulierten Wert (wie in Angriff 1). Aus den Antworten liest er einzelne Divisionsreste des Geheimnisses ab und setzt daraus mithilfe des Chinesischen Restsatzes (CRT) den vollständigen privaten Schlüssel zusammen.

##### Keine Validierung

In [ ]:
// Wähle Bobs statischen privaten Schlüssel (kann beliebig geändert werden, z. B. 13, 7, 22, 29)
BigInteger bobKey = BigInteger.valueOf(13); // <-- Beliebigen Wert testen!
Parameters params = Parameters.createSmall31Bit();
BigInteger p = params.modulus; // 31
System.out.println("Bobs geheimer privater Schlüssel: " + bobKey);

// SITZUNG 1: Angreifer sendet 30 (erzeugt 2er Untergruppe)
System.out.println("\n--- SITZUNG 1 (Angreifer sendet 30, erzeugt 2er Gruppe) ---");
KeyExchange session1 = (new KeyExchange())
    .setParameters(params)
    .setBobPrivateKey(bobKey)
    .setAttackType(KeyExchange.AttackType.ATTACK_3)
    .setValidationMode(KeyExchange.ValidationMode.NONE)
    .setInjectedAttackerValue(BigInteger.valueOf(30));
session1.run();
// DYNAMISCH ERMITTELTER RESTWERT (direkt aus der Sitzung!):
int rest2 = session1.deduceRemainder(2);
System.out.println("-> Angreifer ermittelt aus Sitzung 1: Bob's Private Key % 2 = " + rest2);

// SITZUNG 2: Angreifer sendet 25 (erzeugt 3er Untergruppe)
System.out.println("\n--- SITZUNG 2 (Angreifer sendet 25, erzeugt 3er Gruppe) ---");
KeyExchange session2 = (new KeyExchange())
    .setParameters(params)
    .setBobPrivateKey(bobKey)
    .setAttackType(KeyExchange.AttackType.ATTACK_3)
    .setValidationMode(KeyExchange.ValidationMode.NONE)
    .setInjectedAttackerValue(BigInteger.valueOf(25));
session2.run();
// DYNAMISCH ERMITTELTER RESTWERT:
int rest3 = session2.deduceRemainder(3);
System.out.println("-> Angreifer ermittelt aus Sitzung 2: Bob's Private Key % 3 = " + rest3);

// SITZUNG 3: Angreifer sendet 16 (erzeugt 5er Untergruppe)
System.out.println("\n--- SITZUNG 3 (Angreifer sendet 16, erzeugt 5er Gruppe) ---");
KeyExchange session3 = (new KeyExchange())
    .setParameters(params)
    .setBobPrivateKey(bobKey)
    .setAttackType(KeyExchange.AttackType.ATTACK_3)
    .setValidationMode(KeyExchange.ValidationMode.NONE)
    .setInjectedAttackerValue(BigInteger.valueOf(16));
session3.run();
// DYNAMISCH ERMITTELTER RESTWERT:
int rest5 = session3.deduceRemainder(5);
System.out.println("-> Angreifer ermittelt aus Sitzung 3: Bob's Private Key % 5 = " + rest5);

// FINALE: Chinesischer Restsatz (CRT)
System.out.println("\n [ANGRIFF 3 - FINALE] Chinesischer Restsatz wird angewendet...");
System.out.println("Angreifer weiß nun:");
System.out.println("Bob's Private Key % 2 = " + rest2);
System.out.println("Bob's Private Key % 3 = " + rest3);
System.out.println("Bob's Private Key % 5 = " + rest5);

BigInteger recoveredKey = KeyExchange.solveCRT(new int[]{rest2, rest3, rest5}, new int[]{2, 3, 5});
System.out.println("==> BINGO! Der Angreifer hat Bobs statischen privaten Schlüssel berechnet: " + recoveredKey);
System.out.println("==> Stimmt mit Bobs echtem Schlüssel überein? " + recoveredKey.equals(bobKey));

In [ ]:
// Test mit weiteren Werten für Bobs privaten Schlüssel über simulateAttack3CRT():
System.out.println("=== Test mit Bob's Key = 22 ===");
KeyExchange.simulateAttack3CRT(BigInteger.valueOf(22));

System.out.println("\n=== Test mit Bob's Key = 7 ===");
KeyExchange.simulateAttack3CRT(BigInteger.valueOf(7));


##### Validierung: Öffentliche Werte prüfen
Der Angriff funktioniert nur, solange der Angreifer die öffentlichen Werte manipulieren kann.
Hier kann die Überprüfung der öffentlichen Werte wieder unsichere Werte abfangen, mit denen sich der Angreifer auf geheime Informationen schließen kann.

In [ ]:
Parameters params = Parameters.createSmall31Bit();
BigInteger bobKey = BigInteger.valueOf(13);

System.out.println("\n--- SITZUNG 1 mit Validierung (Angreifer sendet 30) ---");
KeyExchange sessionWithValidation = (new KeyExchange())
    .setParameters(params)
    .setBobPrivateKey(bobKey)
    .setAttackType(KeyExchange.AttackType.ATTACK_3)
    .setValidationMode(KeyExchange.ValidationMode.VALIDATE_PUBLIC_VALUES)
    .setInjectedAttackerValue(BigInteger.valueOf(30));
sessionWithValidation.run();